# MC vs Split Conformal Prediction 시뮬레이션

CLAUDE.md / 시뮬.md 스펙 기준 구현. 한 셀씩 같이 논의하면서 진행.

**현재까지 들어간 것**: RNG 독립 스트림 → 오차분포 3종 → DGP(mean/scale) → 모델 학습(OLS/RF)


In [1]:
import numpy as np
import pandas as pd
from scipy import stats, integrate, optimize
from dataclasses import dataclass
from typing import Callable

np.set_printoptions(precision=6, suppress=True)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")

## 1. RNG (난수 생성기) 독립 스트림

**rng란?** 난수를 차례대로 뽑아주는 객체. 같은 seed로 만들면 언제 실행해도 같은 순서의 값이 나오므로(재현 가능), 실험을 매번 또같이 돌려볼 수 있음.

**왜 하나의 rng를 계속 재사용하면 안 되는가**: 예를 들어 training 데이터와 calibration 데이터를 같은 rng로 순차적으로 만들면, training 쪽 코드를 조금만 고쳐도(난수를 하나라도 더 뽑거나 덜 뽑거나) calibration 데이터까지 달라져버림.

**해결책**: `substream(scenario, b, stage)`처럼 이름표(path)를 붙여서, 이름표가 다르면 서로 완전히 독립적인 rng를 만드는 함수를 만듦. 같은 이름표 → 같은 결과(재현), 다른 이름표 → 서로 무관한 다른 결과(독립).

문자열을 숫자로 바꿀 때는 파이썬 내장 `hash()`를 쓰지 않음(실행마다 값이 바뀌는 경우가 있어 재현성이 깨짐) — 대신 `hashlib.sha256`으로 실행 환경과 무관하게 액상을 고정된 수로 바꿈 (CLAUDE.md §9).

In [2]:
import hashlib

# 이 값이 같으면 전체 실험이 항상 또같이 재현됨
MASTER_SEED = 20240911


def stable_int(value):
    """문자열/정수를 항상 같은 정수로 바꿀 (파이썬 hash()는 실행마다 바뀌어서 사용 금지)."""
    text = str(value)
    digest = hashlib.sha256(text.encode("utf-8")).digest()  # 문자열 -> 32바이트 고정 지문값
    return int.from_bytes(digest[:8], "little")  # 그 중 8바이트만 정수로 변환


def substream(*path):
    """(scenario, b, r, stage, ...) 같은 이름표로 독립적인 rng를 생성.

    같은 path -> 항상 같은 난수열 (재현 가능)
    다른 path -> 서로 독립적인 다른 난수열
    """
    entropy = [MASTER_SEED] + [stable_int(p) for p in path]
    seed_sequence = np.random.SeedSequence(entropy)
    return np.random.default_rng(seed_sequence)

In [3]:
# 간단한 확인: 같은 path는 같은 값, 다른 path(시나리오/stage)는 다른 값
rng_a = substream("linear_homo_gaussian", 1, "train")
rng_b = substream("linear_homo_gaussian", 1, "train")
rng_c = substream("linear_homo_gaussian", 1, "model")  # stage만 다름

print("rng_a :", rng_a.standard_normal(3))
print("rng_b :", rng_b.standard_normal(3))
print("rng_c :", rng_c.standard_normal(3))

assert np.array_equal(rng_a.standard_normal(0), rng_b.standard_normal(0))  # 둘 다 빈 배열, 형태만 확인
print("RNG 기본 성질 확인 완료")


rng_a : [-0.578851  0.592685  1.246618]
rng_b : [-0.578851  0.592685  1.246618]
rng_c : [0.064885 2.014959 1.352782]
RNG 기본 성질 확인 완료


## 2. Error distribution

각 error는 `sample(size, rng)`, `cdf(e)`, `ppf(p)` 세 메서드를 갖는 클래스. 이 세 메서드가 같은 표준화 상수(df, scale, a, s 등)를 공유해야 해서 함수 대신 클래스로 묶음.

- `sample(size, rng)`: 실제 난수 표본을 size개 뽑음
- `cdf(e)`: $P(\varepsilon \le e)$. coverage 계산에 쓰임
- `ppf(p)`: cdf의 역함수(분위수). 예를 들어 `ppf(0.975)`는 상위 2.5% 지점 값

세 분포 모두 $E[\varepsilon]=0$, $\mathrm{Var}(\varepsilon)=1$로 표준화됨:
- Gaussian: 정규분포
- Student-$t$: 대칭이지만 꿀리가 두꺼운 분포
- Lognormal: 비대칭 분포

In [4]:
class GaussianError:
    """E[eps]=0, Var(eps)=1인 표준정규 오차. 표준정규는 원래부터 평균0·분산1이라 별도 변환이 필요 없음."""

    def sample(self, size, rng):
        # 실제 난수를 size개 생성 (몬테카를로 표본, training/calibration의 잡음 등에 사용)
        return rng.standard_normal(size)

    def cdf(self, e):
        # cumulative distribution function: P(eps <= e). coverage 계산에 사용
        return stats.norm.cdf(e)

    def ppf(self, p):
        # cdf의 역함수(quantile function): P(eps <= q) = p 인 q를 반환
        # 예: ppf(0.975) -> 상위 2.5% 지점 값. Oracle 구간의 upper bound 계산에 사용
        return stats.norm.ppf(p)


class StudentTError:
    """eps = T / sqrt(3), T ~ t_3. t_3의 분산이 3이므로 sqrt(3)로 나누어 분산 1로 표준화."""

    def __init__(self, df=3):
        self.df = df  # 자유도(degrees of freedom); 작을수록 꼬리가 두꺼움(heavy tail)
        # t_df 분포의 분산은 df/(df-2). 이 값으로 나눠주면 최종 eps의 분산이 정확히 1이 됨
        self.scale = np.sqrt(df / (df - 2))

    def sample(self, size, rng):
        raw_t_sample = rng.standard_t(self.df, size=size)  # 아직 분산이 1이 아닌 원래 t분포 표본
        return raw_t_sample / self.scale  # scale로 나눠서 분산을 1로 맞춤

    def cdf(self, e):
        # 우리 eps는 원래 t분포를 scale로 나눈 것이므로,
        # 반대로 e에 scale을 다시 곱하면 원래 t분포 값으로 되돌아감 -> 거기서 t분포의 cdf를 사용
        e_in_original_t_scale = np.asarray(e) * self.scale
        return stats.t.cdf(e_in_original_t_scale, df=self.df)

    def ppf(self, p):
        q_in_original_t_scale = stats.t.ppf(p, df=self.df)  # 원래 t분포 기준의 분위수
        return q_in_original_t_scale / self.scale  # 우리 eps 기준(분산 1)으로 다시 변환


class LogNormalError:
    """eps = (exp(Z) - a) / s, Z~N(0,1), a=exp(1/2), s=sqrt(e*(e-1)).

    exp(Z) 자체는 평균이 0도 아니고 분산도 1이 아니므로(로그정규분포),
    평균을 a만큼 빼고 표준편차 s로 나눠서 최종 eps의 평균 0·분산 1을 맞춤.
    """

    def __init__(self):
        self.a = np.exp(0.5)  # exp(Z)의 평균 (Z~N(0,1)일 때 E[exp(Z)] = exp(1/2))
        self.s = np.sqrt(np.e * (np.e - 1))  # exp(Z)의 표준편차

    def sample(self, size, rng):
        z = rng.standard_normal(size)  # 먼저 표준정규 Z를 뽑고
        raw_lognormal = np.exp(z)  # Z를 exp에 넣으면 로그정규분포가 됨 (평균 0·분산 1 아님)
        return (raw_lognormal - self.a) / self.s  # 평균 0·분산 1로 표준화

    def cdf(self, u):
        # 목표: eps <= u 일 확률을 구하는 것
        # 1단계: 우리 eps 기준값 u를, 표준화하기 전의 "원래 exp(Z)" 기준값으로 되돌림
        #        eps = (exp(Z)-a)/s 였으니 반대로 풀면 exp(Z) = s*u + a
        u = np.asarray(u, dtype=float)
        original_scale_value = self.s * u + self.a  # 이게 바로 "s*u+a" = 원래 exp(Z) 기준값

        # 2단계: exp(Z) <= original_scale_value 일 확률을 구하면 됨.
        #        exp(Z)는 항상 양수이므로, original_scale_value <= 0이면 그 확률은 무조건 0
        is_in_support = original_scale_value > 0

        # 3단계: original_scale_value가 0보다 클 때만 log를 취해도 안전함.
        #        0 이하인 곳은 나중에 버릴 값이지만, log(0 이하)는 계산 자체가 에러/경고를 내므로
        #        일단 안전한 더미값 1.0으로 채워서 계산만 통과시킴 (결과는 다음 줄에서 덮어씀)
        safe_value_for_log = np.where(is_in_support, original_scale_value, 1.0)

        # 4단계: exp(Z) <= val  <=>  Z <= log(val) 이므로, 표준정규 cdf(log(val))가 원하는 확률
        probability_if_in_support = stats.norm.cdf(np.log(safe_value_for_log))

        # 5단계: support 밖(원래 값<=0)이었던 자리는 확률 0으로 덮어씀
        result = np.where(is_in_support, probability_if_in_support, 0.0)

        # 6단계: 입력 자체가 NaN이었던 자리는 결과도 NaN으로 유지 (0으로 몰래 바꾸지 않음)
        return np.where(np.isnan(u), np.nan, result)

    def ppf(self, p):
        # cdf의 역순으로 계산: 표준정규 분위수 -> exp로 원래 스케일 -> 다시 (a,s)로 표준화
        p = np.asarray(p, dtype=float)
        standard_normal_quantile = stats.norm.ppf(p)  # Phi^{-1}(p)
        original_scale_value = np.exp(standard_normal_quantile)  # exp(Z) 기준의 분위수
        return (original_scale_value - self.a) / self.s  # 표준화된 eps 기준으로 변환


ERROR_DISTRIBUTIONS = {
    "gaussian": GaussianError(),
    "student_t": StudentTError(),
    "lognormal": LogNormalError(),
}

### 검증: mean≈0, var≈1, cdf(ppf(p))≈p, LogNormal support 경계

In [5]:
rng_check = substream("validation", "error_distribution")
n_check = 2_000_000
p_grid = np.linspace(0.01, 0.99, 25)

for name, err in ERROR_DISTRIBUTIONS.items():
    samples = err.sample(n_check, substream("validation", "error_distribution", name))
    cdf_ppf_err = np.max(np.abs(err.cdf(err.ppf(p_grid)) - p_grid))
    print(f"{name:10s} mean={samples.mean():+.4f}  var={samples.var():.4f}  max|cdf(ppf(p))-p|={cdf_ppf_err:.2e}")

ln = ERROR_DISTRIBUTIONS["lognormal"]
boundary = -ln.a / ln.s
print("\nlognormal support 경계:", "미만", ln.cdf(boundary - 0.01), " / ", "이상", ln.cdf(boundary + 0.01))


gaussian   mean=-0.0003  var=1.0000  max|cdf(ppf(p))-p|=1.11e-16
student_t  mean=+0.0008  var=0.9878  max|cdf(ppf(p))-p|=2.22e-16
lognormal  mean=-0.0011  var=0.9914  max|cdf(ppf(p))-p|=1.11e-16

lognormal support 경계: 미만 0.0  /  이상 6.290800057141992e-05


## 3. DGP (Data Generating Process)

$X=(X_1,X_2)$, $X_1,X_2\overset{iid}{\sim}U(-1,1)$, $Y=m_0(X)+\sigma_0(X)\varepsilon$

- `m0_linear`, `m0_nonlinear`: true conditional mean (평균 0, 분산 4/3으로 통일)
- `sigma0_homo`, `sigma0_hetero`: true conditional scale. hetero는 $X_1$에만 의존
- `draw_X`, `draw_Y`: 설명변수와 반응변수 생성

In [6]:
def m0_linear(X):
    x1, x2 = X[:, 0], X[:, 1]
    return np.sqrt(2) * (x1 + x2)


def m0_nonlinear(X):
    # X1은 비선형(sin), X2는 선형, x1*x2 상호작용 포함.
    # OLS는 절편+X1+X2만 쓰므로 이 경우 OLS에 misspecification(모형 오설정)이 생김
    x1, x2 = X[:, 0], X[:, 1]
    return np.sqrt(6 / 11) * (2 * np.sin(np.pi * x1) + x2 + x1 * x2)


def sigma0_homo(X):
    return np.ones(X.shape[0])  # 모든 위치에서 noise 크기 동일(등분산)


def sigma0_hetero(X):
    x1 = X[:, 0]
    return 0.4 + 1.2 * np.sin(np.pi * x1) ** 2  # X1에 따라 noise 크기가 달라짐(이분산)


MEAN_FUNCTIONS = {"linear": m0_linear, "nonlinear": m0_nonlinear}
SCALE_FUNCTIONS = {"homo": sigma0_homo, "hetero": sigma0_hetero}


def draw_X(n, rng):
    return rng.uniform(-1.0, 1.0, size=(n, 2))  # (n, 2) shape, 각 열이 U(-1,1)에서 독립


def draw_Y(X, mean_fn, scale_fn, error, rng):
    eps = error.sample(X.shape[0], rng)  # X와 독립적인 잡음
    return mean_fn(X) + scale_fn(X) * eps


### 검증: mean 평균 0·분산 4/3, scale 최소 양수·평균제곱 1(homo)/1.18(hetero)

In [7]:
rng_dgp_check = substream("validation", "dgp")
n_check = 5_000_000
X_check = draw_X(n_check, rng_dgp_check)

for name, fn in MEAN_FUNCTIONS.items():
    m = fn(X_check)
    print(f"m0_{name:10s} mean={m.mean():+.5f} (target 0)   var={m.var():.5f} (target {4/3:.4f})")

for name, fn in SCALE_FUNCTIONS.items():
    s = fn(X_check)
    target = 1.0 if name == "homo" else 1.18
    print(f"sigma0_{name:8s} min={s.min():.5f} (>0)   mean(sigma^2)={np.mean(s**2):.5f} (target {target})")


m0_linear     mean=+0.00049 (target 0)   var=1.33376 (target 1.3333)
m0_nonlinear  mean=+0.00068 (target 0)   var=1.33376 (target 1.3333)
sigma0_homo     min=1.00000 (>0)   mean(sigma^2)=1.00000 (target 1.0)
sigma0_hetero   min=0.40000 (>0)   mean(sigma^2)=1.18016 (target 1.18)


### DGP 컨테이너

`mean_fn`, `scale_fn`, `error`를 하나의 시나리오로 묶음. 2 mean × 2 scale × 3 error = 12개 DGP.

In [8]:
@dataclass
class DGP:
    name: str
    mean_fn: Callable
    scale_fn: Callable
    error: object
    error_name: str  # rng_mc를 error별로 공유할 때 쓸 이름 (DGP와 불일치 방지용)
    scale_name: str  # q0(res_true 반폭)가 mean과 무관하게 scale·error에만 의존해서 캐시할 때 쓸 이름

    def sample_xy(self, n, rng):
        X = draw_X(n, rng)
        Y = draw_Y(X, self.mean_fn, self.scale_fn, self.error, rng)
        return X, Y


DGPS = {}
for mean_name, mean_fn in MEAN_FUNCTIONS.items():
    for scale_name, scale_fn in SCALE_FUNCTIONS.items():
        for error_name, error in ERROR_DISTRIBUTIONS.items():
            dgp_name = f"{mean_name}_{scale_name}_{error_name}"
            DGPS[dgp_name] = DGP(dgp_name, mean_fn, scale_fn, error, error_name, scale_name)

print(f"총 {len(DGPS)}개 DGP")
DGPS

총 12개 DGP


{'linear_homo_gaussian': DGP(name='linear_homo_gaussian', mean_fn=<function m0_linear at 0x000001D23EF1BF60>, scale_fn=<function sigma0_homo at 0x000001D23EC8BBA0>, error=<__main__.GaussianError object at 0x000001D21F58B620>, error_name='gaussian', scale_name='homo'),
 'linear_homo_student_t': DGP(name='linear_homo_student_t', mean_fn=<function m0_linear at 0x000001D23EF1BF60>, scale_fn=<function sigma0_homo at 0x000001D23EC8BBA0>, error=<__main__.StudentTError object at 0x000001D21F58B4D0>, error_name='student_t', scale_name='homo'),
 'linear_homo_lognormal': DGP(name='linear_homo_lognormal', mean_fn=<function m0_linear at 0x000001D23EF1BF60>, scale_fn=<function sigma0_homo at 0x000001D23EC8BBA0>, error=<__main__.LogNormalError object at 0x000001D21F58B770>, error_name='lognormal', scale_name='homo'),
 'linear_hetero_gaussian': DGP(name='linear_hetero_gaussian', mean_fn=<function m0_linear at 0x000001D23EF1BF60>, scale_fn=<function sigma0_hetero at 0x000001D21FA96D40>, error=<__main__

## 4. 모델 학습 (OLS, RF)

- OLS: 절편 + $X_1,X_2$만 사용
- RF: squared-error 회귀 forest, 1000 trees. 나머지 하이퍼파라미터는 sklearn 기본값을 그대로 쓰을 pilot 기본안으로 명시

In [9]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

RF_HYPERPARAMS = dict(
    n_estimators=1000,
    criterion="squared_error",
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features=1.0,
    bootstrap=True,
    n_jobs=1,  # 밖에서 b(training 반복)를 병렬화할 예정이라 RF 내부 병렬화는 꺼서 oversubscription 방지
)


def fit_ols(X, y):
    return LinearRegression().fit(X, y)


def fit_rf(X, y, rng):
    seed = int(rng.integers(0, 2**31 - 1))  # rng에서 정수 하나를 뽑아 sklearn의 random_state로 사용
    return RandomForestRegressor(random_state=seed, **RF_HYPERPARAMS).fit(X, y)


### 확인: linear_homo_gaussian에서 학습해보기

$n_{train}=1000$으로 training 데이터를 만들고 OLS·RF를 학습. 데이터 생성용 rng와 RF 학습용 rng를 다른 stage로 분리해서 독립적으로 쓰는다.

In [10]:
dgp = DGPS["linear_homo_gaussian"]
b = 1

rng_train = substream(dgp.name, b, "train")
rng_model = substream(dgp.name, b, "model")

X_train, y_train = dgp.sample_xy(1000, rng_train)
ols_model = fit_ols(X_train, y_train)
rf_model = fit_rf(X_train, y_train, rng_model)

print("OLS intercept:", ols_model.intercept_, "(target 0)")
print("OLS coef     :", ols_model.coef_, f"(target both {np.sqrt(2):.4f})")

x_eval = np.array([[0.0, 0.0], [0.5, -0.5], [0.9, 0.9]])
print("\ntrue m0    :", dgp.mean_fn(x_eval))
print("OLS predict:", ols_model.predict(x_eval))
print("RF  predict:", rf_model.predict(x_eval))

OLS intercept: -0.015698804450583234 (target 0)
OLS coef     : [1.345043 1.456035] (target both 1.4142)

true m0    : [0.       0.       2.545584]
OLS predict: [-0.015699 -0.071195  2.505271]
RF  predict: [-0.309961 -0.518883  2.387691]


## 5. Oracle, MC 구간

- `oracle_interval`: true 오차분포의 분위수로 정확히 계산한 구간. $[m_0(x)+\sigma_0(x)q_{\alpha/2},\ m_0(x)+\sigma_0(x)q_{1-\alpha/2}]$
- `mc_interval`: 실제로 오차 표본을 $M_{MC}$개 뽑아서, 그 표본의 empirical quantile로 만든 구간 (oracle을 몬테카를로로 근사)
- `conditional_coverage`: 구간 $[L,U]$가 특정 위치 $x$에서 새 $Y$를 포함할 확률. $\{L < Y<U \}=\{L<m_0(x)+\sigma_0(x)\epsilon<U\}=\{\frac{L-m_0(x)}{\sigma_0(x)}<\epsilon<\frac{U-m_0(x)}{\sigma_0(x)}\}\Rightarrow F_\varepsilon\!\left(\frac{U-m_0(x)}{\sigma_0(x)}\right)-F_\varepsilon\!\left(\frac{L-m_0(x)}{\sigma_0(x)}\right)$ — CDF로 직접 계산하므로 새 $Y$를 뽑을 필요 없음 (CLAUDE.md §6.1, §9)

MC 표본(`eps_sample`)은 error 분포에서만 뽑고, mean/scale은 그 뒤에 위치·크기 변환으로 적용 — 그래서 같은 표본을 모든 $x$에 재사용 가능 (시뮬.md §4.3).

In [11]:
def oracle_interval(X, dgp, alpha):
    """true 분위수를 그대로 사용해서 이론적으로 정확한 구간을 만든다.

    구간 = [m0(x) + sigma0(x)*q_lo, m0(x) + sigma0(x)*q_hi]
    여기서 q_lo, q_hi는 진짜 오차분포 eps의 양쪽 분위수.
    """
    # 1단계: 이 x에서 true mean, true scale을 계산 (우리가 DGP를 직접 설계했으니 정확히 알고 있음)
    true_mean = dgp.mean_fn(X)
    true_scale = dgp.scale_fn(X)

    # 2단계: 목표 alpha에 맞는 양쪽 분위수를 구함
    #        alpha=0.05면 q_lo=하위 2.5% 지점, q_hi=상위 97.5% 지점(=하위 2.5%를 뺀 지점)
    q_lo = dgp.error.ppf(alpha / 2)
    q_hi = dgp.error.ppf(1 - alpha / 2)

    # 3단계: 위치(true_mean)에 폭(true_scale * 분위수)을 더해서 구간의 양 끝을 만듦
    lower_bound = true_mean + true_scale * q_lo
    upper_bound = true_mean + true_scale * q_hi
    return lower_bound, upper_bound


def draw_mc_benchmark(dgp, M_MC, rng):
    """MC 구간을 만드는 데 쓸 오차 표본을 M_MC개 뽑는다.

    이 표본은 error 분포에서만 뽑히고 mean/scale과는 무관하므로,
    같은 표본을 여러 x 위치에 재사용할 수 있다 (mc_interval에서 위치·크기 변환만 적용).
    """
    return dgp.error.sample(M_MC, rng)


def mc_interval(X, dgp, alpha, eps_sample):
    """오차 표본(eps_sample)의 empirical quantile로 oracle 구간을 몬테카를로 근사한다."""
    # 1단계: 표본에서 empirical quantile(경험적 분위수)을 계산.
    #        method="linear"는 numpy의 기본 보간 방식 -> 이 보간 규칙을 고정해서 기록해둠
    q_lo, q_hi = np.quantile(eps_sample, [alpha / 2, 1 - alpha / 2], method="linear")

    # 2단계: oracle_interval과 똑같은 방식으로, true mean/scale에 이 분위수를 적용
    true_mean = dgp.mean_fn(X)
    true_scale = dgp.scale_fn(X)
    lower_bound = true_mean + true_scale * q_lo
    upper_bound = true_mean + true_scale * q_hi
    return lower_bound, upper_bound


def conditional_coverage(L, U, X, dgp):
    """구간 [L,U]가 특정 위치 x에서 실제로 새 Y를 포함할 확률.

    새로운 Y를 무작위로 많이 뽑아서 세는 방식이 아니라, true CDF로 정확히 계산한다
    (CLAUDE.md §6.1, §9: coverage는 CDF로 직접 계산하고 이진 포함 횟수 추정으로 대체하지 않음).
    """
    true_mean = dgp.mean_fn(X)
    true_scale = dgp.scale_fn(X)

    # Y = m0(X) + sigma0(X)*eps 이므로, L <= Y <= U  <=>  (L-m0)/sigma0 <= eps <= (U-m0)/sigma0
    # 즉 끝점 L, U를 "표준화된 eps 기준"으로 바꾼 뒤, 그 사이에 eps가 들어올 확률을 CDF 차이로 구함
    standardized_lower = (L - true_mean) / true_scale
    standardized_upper = (U - true_mean) / true_scale
    return dgp.error.cdf(standardized_upper) - dgp.error.cdf(standardized_lower)

### 확인: oracle vs mc, 같은 x에서 비교

In [12]:
dgp = DGPS["nonlinear_hetero_lognormal"]  # 가장 어려운 조합(이분산 + 비대칭 오차)으로 확인
alpha = 0.05
M_MC = 100_000

eps_sample = draw_mc_benchmark(dgp, M_MC, substream("mc_benchmark", dgp.error_name))

x_eval = np.array([[0.0, 0.0], [0.9, 0.9]])
L_oracle, U_oracle = oracle_interval(x_eval, dgp, alpha)
L_mc, U_mc = mc_interval(x_eval, dgp, alpha, eps_sample)

print("oracle:", np.stack([L_oracle, U_oracle], axis=1))
print("mc    :", np.stack([L_mc, U_mc], axis=1))
print("endpoint 차이 (L,U):", L_mc - L_oracle, U_mc - U_oracle)
print("oracle coverage:", conditional_coverage(L_oracle, U_oracle, x_eval, dgp), "(target 0.95)")
print("mc     coverage:", conditional_coverage(L_mc, U_mc, x_eval, dgp))

oracle: [[-0.279078  1.008765]
 [ 1.36034   3.017117]]
mc    : [[-0.27933   1.005121]
 [ 1.360016  3.01243 ]]
endpoint 차이 (L,U): [-0.000252 -0.000324] [-0.003644 -0.004688]
oracle coverage: [0.95 0.95] (target 0.95)
mc     coverage: [0.950399 0.950399]


### $M_{MC}$ 탐색: MC 분위수가 true 분위수에 얼마나 가까워지는가

MC 구간의 정확도는 mean/scale과 무관하게 **error 분포 + $M_{MC}$ + $\alpha$**로만 결정됨 (mean/scale은 나중에 곱하고 더하는 변환일 뿐이라서). 그래서 raw 분위수 오차 $|\hat q - q_{true}|$를 직접 보는 게 가장 간단.

$\alpha=0.01$(양쪽 0.5%)처럼 극단적인 분위수, 그리고 Student-$t$·LogNormal처럼 꼬리가 무겁거나 비대칭인 분포가 가장 느리게 수렴할 것으로 예상 — 이 조합으로 $M_{MC}$를 늘려가며 확인.

In [13]:
M_MC_CANDIDATES = [1_000, 10_000, 100_000, 1_000_000, 10_000_000]
ALPHAS = [0.10, 0.05, 0.01]
N_REPS = 5  # 같은 M_MC라도 반복마다 오차가 얼마나 흔들리는지 보기 위해 여러 번 반복

rows = []  # 표로 만들 결과를 한 줄씩 여기에 쌓음

# 바깥 루프: error 분포 3종류를 하나씩 확인
for error_name, err in ERROR_DISTRIBUTIONS.items():
    # 그 다음 루프: alpha(목표 coverage 수준) 3종류를 하나씩 확인
    for alpha in ALPHAS:
        # 이 error, 이 alpha에서의 "정답" 분위수 (M_MC와 무관하게 고정된 진짜 값)
        q_lo_true = err.ppf(alpha / 2)
        q_hi_true = err.ppf(1 - alpha / 2)

        # 그 다음 루프: M_MC(표본 개수)를 늘려가며 확인
        for M_MC in M_MC_CANDIDATES:
            worst_err = 0.0  # 이 (error, alpha, M_MC) 조합에서 N_REPS번 중 가장 컸던 오차를 기록

            # 가장 안쪽 루프: 같은 조건을 N_REPS번 반복해서, 운이 좋아서 우연히 잘 맞은 건 아닌지 확인
            for rep in range(N_REPS):
                rng = substream("mc_pilot", error_name, alpha, M_MC, rep)  # 반복마다 독립적인 rng
                sample = err.sample(M_MC, rng)  # 실제로 M_MC개의 오차 표본을 뽑음
                q_lo, q_hi = np.quantile(sample, [alpha / 2, 1 - alpha / 2], method="linear")

                # 이번 반복에서의 오차(추정값과 정답의 차이)를 계산해서, 지금까지의 최댓값과 비교
                error_lo = abs(q_lo - q_lo_true)
                error_hi = abs(q_hi - q_hi_true)
                worst_err = max(worst_err, error_lo, error_hi)

            rows.append({"error": error_name, "alpha": alpha, "M_MC": M_MC, "worst_err": worst_err})

# 쌓아둔 결과를 표로 만듦: error x alpha를 행으로, M_MC를 열로 펼쳐서 한눈에 비교
mc_error_table = pd.DataFrame(rows).pivot(index=["error", "alpha"], columns="M_MC", values="worst_err")
mc_error_table

M_MC              1000      10000     100000    1000000   10000000
error     alpha                                                   
gaussian  0.0100    0.3719    0.1169    0.0365    0.0132    0.0025
          0.0500    0.1700    0.0408    0.0152    0.0032    0.0018
          0.1000    0.0798    0.0381    0.0219    0.0058    0.0011
lognormal 0.0100    1.1907    0.7288    0.0915    0.0617    0.0121
          0.0500    0.5504    0.1400    0.0369    0.0072    0.0063
          0.1000    0.3340    0.0860    0.0323    0.0084    0.0047
student_t 0.0100    1.1262    0.2405    0.1048    0.0276    0.0128
          0.0500    0.2509    0.0760    0.0239    0.0101    0.0020
          0.1000    0.2327    0.0508    0.0132    0.0051    0.0020

## 6. `res_true` 구간 ($q_{0,\alpha}$: population residual quantile)

True mean $m_0$은 그대로 쓰지만, 폭은 **전체 $X$ 분포에서 나온 residual** $R_0=|Y-m_0(X)|=|\sigma_0(X)\varepsilon|$의 $(1-\alpha)$ population quantile $q_{0,\alpha}$ 하나로 고정(CP 방식으로 score는 계산하는데, fitted model 대신 true model을 쓰고, finite calibration 대신 무한 calibration population을 쓴 것):

$$C_{res,0}(x) = [m_0(x)-q_{0,\alpha},\ m_0(x)+q_{0,\alpha}]$$

$q_{0,\alpha}$를 구하는 식: $H(q) = E_X\!\left[F_\varepsilon(q/\sigma_0(X)) - F_\varepsilon(-q/\sigma_0(X))\right] = 1-\alpha$

- $\sigma_0(X)$가 $X_1$에만 의존하므로(homo·hetero 둘 다), $H(q)$는 **$X_1$에 대한 1차원 적분**으로 정확히 계산 가능 (`scipy.integrate.quad`)
- 그 적분값이 목표($1-\alpha$)와 같아지는 $q$를 root-finding으로 찾음 (`scipy.optimize.brentq`)
- $q_{0,\alpha}$는 **mean과 무관** — scale·error·alpha가 같으면 항상 같은 값이라 캐시 가능

In [14]:
# (scale_name, error_name, alpha) -> 이미 계산해둔 q0 값. mean과는 무관하므로 DGP 이름 전체가 아니라
# 이 세 가지로만 캐시하면 됨 (같은 scale/error/alpha면 mean이 달라도 항상 같은 q0)

# q 후보 선택 → H(q) 계산 → 1-α와 비교 → q 조정
_Q0_CACHE = {}

def _probability_within_q(q, x1, scale_fn, error):
    """위치 x1 하나에서, |R0| <= q 일 확률을 계산하는 가장 작은 단위 함수.

    R0 = |Y - m0(X)| = |sigma0(X) * eps| 이므로,
    |R0| <= q  <=>  -q/sigma0(X) <= eps <= q/sigma0(X)
    이 확률을 CDF 차이로 구한다.
    """
    # scale_fn은 (n,2) 모양의 X를 받아서 X[:,0](=x1)만 사용하므로, x2 자리는 0으로 채워도 결과에 영향 없음
    X_dummy = np.array([[x1, 0.0]])
    sigma0_at_x1 = scale_fn(X_dummy)[0]

    upper_side = error.cdf(q / sigma0_at_x1)
    lower_side = error.cdf(-q / sigma0_at_x1)
    return upper_side - lower_side


def _H_res_true(q, scale_fn, error):
    """H(q) = E_X1[ |R0| <= q 일 확률 ]=int_{-1}^{1} f(X_1)P(|R_0| ≤ q)dX_1. X1을 모든 위치에서 평균낸 것.

    X1 ~ U(-1,1)이므로 pdf가 항상 1/2. scipy.integrate.quad로 -1~1 구간을 적분해서
    "모든 X1에서의 확률"을 정확하게(수치적분으로) 평균낸다.
    """
    def integrand(x1):
        probability_at_this_x1 = _probability_within_q(q, x1, scale_fn, error)
        return probability_at_this_x1 * 0.5  # U(-1,1)의 밀도 1/2을 곱해야 진짜 적분(평균)이 됨

    integral_value, _ = integrate.quad(integrand, -1, 1)
    return integral_value


def compute_q0(scale_name, error_name, alpha):
    """H(q) = 1-alpha 를 만족하는 q0를 찾는다 (res_true 구간의 반폭).

    H(q)는 q가 커질수록 커지는(단조증가) 함수이므로 -- q=0이면 확률 0, q가 아주 크면 확률 1에 가까움 --
    이분법 계열의 root-finding(brentq)으로 정확히 그 값을 찾을 수 있다.
    """
    cache_key = (scale_name, error_name, alpha)
    if cache_key in _Q0_CACHE:
        return _Q0_CACHE[cache_key]  # 이미 계산해둔 값이면 다시 계산하지 않고 재사용

    scale_fn = SCALE_FUNCTIONS[scale_name]
    error = ERROR_DISTRIBUTIONS[error_name]
    target_probability = 1 - alpha

    # H(q) - target = 0 이 되는 q를 찾는 것. brentq(함수, 하한, 상한)은
    # "하한에서는 함수값이 음수, 상한에서는 양수(또는 그 반대)"인 구간 안에서 정확한 근을 찾아준다.
    # q=1e-8(거의 0)이면 H≈0 이라서 target보다 작고, q=50이면 H≈1이라서 target보다 크므로 근이 이 사이에 있음이 보장됨.
    def h_minus_target(q):
        return _H_res_true(q, scale_fn, error) - target_probability

    q0 = optimize.brentq(h_minus_target, 1e-8, 50.0, xtol=1e-10)

    _Q0_CACHE[cache_key] = q0  # 다음에 같은 (scale, error, alpha) 조합이 오면 재사용
    return q0


def res_true_interval(X, dgp, alpha):
    """[m0(x) - q0, m0(x) + q0]. true mean은 그대로 쓰고, 폭만 전체 X에서 나온 q0로 고정."""
    q0 = compute_q0(dgp.scale_name, dgp.error_name, alpha)
    true_mean = dgp.mean_fn(X)
    lower_bound = true_mean - q0
    upper_bound = true_mean + q0
    return lower_bound, upper_bound

### 검증: q0가 맞게 계산되는지 두 가지 방법으로 확인

1. **homo + 대칭분포(gaussian, student_t)에서는 $q_0=$ true 분위수와 정확히 같아야 함** — $\sigma_0\equiv1$이면 $R_0=|\varepsilon|$이라서, $q_{0,\alpha}$는 그냥 $\varepsilon$의 $(1-\alpha)$ 분위수. CLAUDE.md §11 "대칭·등분산에서 Oracle=res_true" 검증
2. **hetero는 독립적인 대량 Monte Carlo 표본과 비교** — 적분/root-finding 결과가 실제 표본 분위수와 일치하는지 확인

In [15]:
# 검증 1) homo에서는 q0 == error.ppf(1-alpha/2) 이어야 함 (대칭분포일 때).
#          이유: sigma0_homo(x)=1이라서 R0=|Y-m0(X)|=|eps| 그 자체이고,
#          대칭분포는 상위 (1-alpha/2) 분위수와 -하위(alpha/2) 분위수가 같으므로
#          |eps|의 (1-alpha) 분위수는 그냥 eps의 (1-alpha/2) 분위수와 같아짐.
print("[homo 검증] q0 vs true ppf(1-alpha/2)")
for error_name in ["gaussian", "student_t"]:
    error = ERROR_DISTRIBUTIONS[error_name]
    for alpha in [0.10, 0.05, 0.01]:
        q0 = compute_q0("homo", error_name, alpha)
        q_true = error.ppf(1 - alpha / 2)
        diff = abs(q0 - q_true)
        print(f"  {error_name:10s} alpha={alpha:<5} q0={q0:.6f}  true={q_true:.6f}  diff={diff:.2e}")

# 검증 2) hetero는 이론값이 따로 없으므로, 독립적으로 뽑은 대량 MC 표본의 분위수와 비교.
#          적분+root-finding으로 구한 q0와, 실제 표본에서 직접 잰 분위수가 비슷해야 함.
print("\n[hetero 검증] q0 vs 독립 MC 표본 분위수 (n=20,000,000)")

n_mc = 20_000_000
# 검증 전용 rng로 X1을 뽑음 (본 실험의 데이터 생성 rng와 완전히 분리된 스트림)
x1_check = draw_X(n_mc, substream("validation", "q0_hetero"))[:, 0]  # X2는 sigma0_hetero가 안 쓰므로 무시
X_check_2d = np.column_stack([x1_check, np.zeros(n_mc)])  # sigma0_hetero(X)가 (n,2) 모양을 기대하므로 맞춰줌
sigma0_check = sigma0_hetero(X_check_2d)

for error_name in ["gaussian", "student_t", "lognormal"]:
    error = ERROR_DISTRIBUTIONS[error_name]
    eps_check = error.sample(n_mc, substream("validation", "q0_hetero", error_name))
    R0_check = np.abs(sigma0_check * eps_check)  # R0 = |Y - m0(X)| = |sigma0(X) * eps|

    for alpha in [0.10, 0.05, 0.01]:
        q0_from_integral = compute_q0("hetero", error_name, alpha)
        q0_from_mc_sample = np.quantile(R0_check, 1 - alpha)
        diff = abs(q0_from_integral - q0_from_mc_sample)
        print(
            f"  {error_name:10s} alpha={alpha:<5} "
            f"q0(적분)={q0_from_integral:.6f}  q0(MC)={q0_from_mc_sample:.6f}  diff={diff:.2e}"
        )

[homo 검증] q0 vs true ppf(1-alpha/2)
  gaussian   alpha=0.1   q0=1.644854  true=1.644854  diff=1.40e-12
  gaussian   alpha=0.05  q0=1.959964  true=1.959964  diff=3.33e-15
  gaussian   alpha=0.01  q0=2.575829  true=2.575829  diff=1.33e-15
  student_t  alpha=0.1   q0=1.358715  true=1.358715  diff=5.34e-13
  student_t  alpha=0.05  q0=1.837386  true=1.837386  diff=8.88e-16
  student_t  alpha=0.01  q0=3.372251  true=3.372251  diff=2.04e-11

[hetero 검증] q0 vs 독립 MC 표본 분위수 (n=20,000,000)
  gaussian   alpha=0.1   q0(적분)=1.818114  q0(MC)=1.817921  diff=1.94e-04
  gaussian   alpha=0.05  q0(적분)=2.310716  q0(MC)=2.310791  diff=7.51e-05
  gaussian   alpha=0.01  q0(적분)=3.314805  q0(MC)=3.315089  diff=2.84e-04
  student_t  alpha=0.1   q0(적분)=1.443203  q0(MC)=1.443823  diff=6.20e-04
  student_t  alpha=0.05  q0(적분)=2.017706  q0(MC)=2.017485  diff=2.21e-04
  student_t  alpha=0.01  q0(적분)=3.831315  q0(MC)=3.833006  diff=1.69e-03
  lognormal  alpha=0.1   q0(적분)=1.020377  q0(MC)=1.020757  diff=3.81e-04
  lo

**결과 해석**: 표를 보면 $\alpha$가 작을수록(0.01) 극단 분위수라 수렴이 느리고, $\alpha=0.10$은 상대적으로 빠르게 안정됨. Gaussian은 $10^5$ 정도면 세 alpha 모두 오차가 충분히 작아짐. Student-t·LogNormal은 $\alpha=0.01$에서 훨씬 느리게 줄어들어서, $10^6$으로도 부족하고 $10^7$까지 가야 오차가 한 자릿수 더 줄어듦. 연산 시간은 $10^7$도 몇 초면 끝나서 문제 없음.

**$M_{MC}$ 후보 결정에 이 표를 어떻게 쓰는가**: 나중에 structural/fitting/calibration 분해에서 보려는 차이의 크기(예: 0.01~0.05 수준)보다 이 MC 오차가 충분히 작아야 함. 표에서 그 기준을 넘기는 가장 작은 $M_{MC}$를 고르면 됨 — 지금 수치로는 **$M_{MC}=10^7$** 정도가 가장 어려운 경우(Student-t·LogNormal, $\alpha=0.01$)에서도 안전해 보임 (pilot 제안값, 최종 확정 아님).

## 현재 구현 범위

구현되고 검증됨: RNG 독립 스트림, 오차분포 3종, DGP, OLS/RF 학습, oracle·mc 구간 + M_MC 탐색, **res_true 구간(q0, 1차원 적분+root-finding)**.

아직 구현 안 된 것:
- 나머지 2개 구간 ($C_{pop,j}, C_{CP,j}$)
- marginal·fixed-x·fixed-calibration 반복 구조
- 중심·폭 평가와 3단계 분해, 결과 저장